# M1 · Step 3 — Feature Engineering Walkthrough

Story: take the cleaned MCF posting frame produced in step 1
(`../data/clean_job_step1.pkl`, 1,012,928 rows × 21 columns), run it
through `feature_engineering.build_features`, and produce the
67-column feature frame the Streamlit dashboard consumes.

**Audience:** anyone reviewing this assignment. Each cell narrates
*why* a block exists, *what* it derives, and *which downstream
dashboard widget consumes it*.

Sections:
1. Load + sanity-check the raw frame
2. Walk each pipeline block on a small sample
3. Run the full pipeline
4. Validate the output (shape, NaN audit, within-category z-scores)
5. Save the output for the dashboard

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from feature_engineering import (
    add_time_features, add_salary_features, add_engagement_features,
    add_title_features, add_company_features, add_quality_flags,
    add_normalized_features, add_recency_features,
    add_company_enrichments, add_composite_scores, build_features,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

## 1. Load + sanity-check the raw frame

The cleaned step-1 pickle is the single input. 21 columns, ~1M rows.
All downstream features derive from these.

In [ ]:
RAW = Path('../data/clean_job_step1.pkl')
df_raw = pd.read_pickle(RAW)

print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw["metadata_originalPostingDate"].min()} '
      f'to {df_raw["metadata_originalPostingDate"].max()}')
print(f'Unique companies: {df_raw["postedCompany_name"].nunique():,}')
print(f'Unique categories: {df_raw["category_1"].nunique()}')
df_raw.dtypes

In [ ]:
# A small stratified sample makes the per-block exploration fast.
# Stratifying by category preserves the mix the dashboard cares about.
sample = (
    df_raw.groupby('category_1', observed=True, group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 200), random_state=0))
    .reset_index(drop=True)
)
print(f'Sample shape: {sample.shape}')
sample.head(3).T

## 2. Walk each pipeline block

Every `add_*_features` function returns a new DataFrame — no
in-place mutation. That lets us inspect the deltas one block at a
time.

### 2.1 Time / lifecycle features

Derives posting duration, repost lag, posting year/month/quarter,
day-of-week, and the `is_reposted` flag. Powers Page 1's median-
duration KPI and the time-cohort filters on the sidebar.

In [ ]:
s1 = add_time_features(sample)
new = [c for c in s1.columns if c not in sample.columns]
print('Added:', new)
s1[new].head()

### 2.2 Salary features

Salary range, normalised spread, S$-band bucketing, and pay per
year of experience. The bands (`<3k, 3-5k, 5-8k, 8-12k, 12-20k, 20k+`)
drive the salary-band sidebar filter and Page 1's salary distribution
chart.

In [ ]:
s2 = add_salary_features(s1)
new = [c for c in s2.columns if c not in s1.columns]
print('Added:', new)
print(s2['salary_band'].value_counts().sort_index())
s2[new].head()

### 2.3 Engagement features

Applications per view, applications per vacancy, views per vacancy,
and a `low_engagement_flag` (high views + low conversion). Powers
Page 2's engagement-funnel scatter and the hidden-gems table.

In [ ]:
s3 = add_engagement_features(s2)
new = [c for c in s3.columns if c not in s2.columns]
print('Added:', new)
s3[new].describe()

### 2.4 Title features

Regex-based seniority bucketing (C-suite → Junior) plus title
word/char counts. The `title_seniority` ordered categorical drives
Page 1's seniority donut, the sidebar filter, and Page 2's
demand-intensity heatmap row dimension.

In [ ]:
s4 = add_title_features(s3)
print(s4['title_seniority'].value_counts())
s4[['title', 'title_seniority']].sample(8, random_state=1)

### 2.5 Company features

Per-company posting count, average salary, agency-vs-direct flag
(keyword-matched on company name), posting-volume bucket. `is_agency`
drives the employer-type radio in the sidebar and the colour split
on Page 1's top-companies chart.

In [ ]:
s5 = add_company_features(s4)
print(f"Agency share in sample: {s5['is_agency'].mean():.1%}")
s5[['postedCompany_name', 'is_agency', 'company_posting_count',
    'company_posting_bucket']].drop_duplicates().head(10)

### 2.6 Quality flags

Six boolean flags. **Used as sidebar exclusions** so recruiters
can drop noise before benchmarking:

- `salary_undisclosed_flag` (min == max — ~0% on this dataset)
- `salary_suspicious_low` (< S$1,500)
- `mass_hiring_flag` (`numberOfVacancies > 10`, ~1.5%)
- `zero_engagement_flag` (no views AND no applications)
- `seniority_mismatch_flag` (title-derived vs positionLevels disagree)

Page 3 §3.2 ("Promising roles") uses several of these in its
exclusion criteria.

In [ ]:
s6 = add_quality_flags(s5)
flags = ['salary_undisclosed_flag', 'salary_suspicious_low',
         'mass_hiring_flag', 'zero_engagement_flag',
         'seniority_mismatch_flag']
s6[flags].mean().to_frame('share').style.format('{:.2%}')

### 2.7 Within-category normalisation

**The magic block.** Compares each row's salary and views to peers
in the same `category_1`. Drives Page 2's salary positioning section
and the percentile-based promising-role criterion on Page 3.

- `salary_zscore_within_category` — "this role pays 1.2σ above
  category mean"
- `salary_percentile_within_category` — "bottom 20% / top 25%"
- `salary_premium_vs_category_median` — raw S$ delta
- `views_zscore_within_category`, `apps_per_view_vs_category_median`

In [ ]:
s7 = add_normalized_features(s6)
new = [c for c in s7.columns if c not in s6.columns]
print('Added:', new)
s7[new].describe()

### 2.8 Recency / cohort features

`days_since_posting` (anchored at the snapshot date), 
`posting_cohort_quarter`, `is_active_at_snapshot`. The snapshot
defaults to the latest `metadata_originalPostingDate` so the dataset
is reproducible as an "as-of" view.

In [ ]:
s8 = add_recency_features(s7)
new = [c for c in s8.columns if c not in s7.columns]
print('Added:', new)
s8[new].head()

### 2.9 + 2.10 Company enrichments + composite scores

Deeper employer rollups (category diversity, salary spread, tenure)
and the three headline composite scores:

- `hard_to_fill_score` — duration + reposts + inverse apps/vacancy
- `demand_intensity_score` — z-scored apps/vacancy
- `posting_quality_score` — views + conversion − duration

All three are z-scored to mean 0, std 1, clipped to ±5. `>1.0` means
notably above market; `<-1.0` notably below.

In [ ]:
s9 = add_company_enrichments(s8)
s10 = add_composite_scores(s9)
scores = ['hard_to_fill_score', 'demand_intensity_score', 'posting_quality_score']
s10[scores].describe().loc[['mean', 'std', 'min', 'max']].round(2)

## 3. Run the full pipeline

All ten blocks in order on the full 1M-row frame. The orchestrator
`build_features` is exactly the concatenation we just walked through.

In [ ]:
t0 = time.time()
df = build_features(df_raw)
elapsed = time.time() - t0

print(f'Built {df.shape[0]:,} rows × {df.shape[1]} columns in {elapsed:.1f}s')
print(f'Raw columns: {df_raw.shape[1]}')
print(f'Derived columns: {df.shape[1] - df_raw.shape[1]}')

## 4. Validate

Three checks from `skills.md`:
1. Shape — 67 columns expected.
2. NaN audit — only the engagement features (which divide by zero-
   guarded denominators) should have NaNs.
3. Within-category z-scores — by construction, each category's
   `salary_zscore_within_category` should have mean ≈ 0, std ≈ 1.

In [ ]:
assert df.shape[1] == 67, f'Expected 67 columns, got {df.shape[1]}'
print('✅ Shape check passed:', df.shape)

In [ ]:
nan_share = df.isna().mean().sort_values(ascending=False)
nan_share[nan_share > 0].to_frame('NaN share').style.format('{:.2%}')

In [ ]:
z_check = (
    df.groupby('category_1', observed=True)['salary_zscore_within_category']
    .agg(['mean', 'std'])
    .round(3)
)
print('Each category should have mean≈0, std≈1:')
z_check.head(10)

## 5. Save for the dashboard

The Streamlit app reads `data/mcf_features.pkl` via
`load_features()` (cached). Re-run this cell whenever the raw data
refreshes.

In [ ]:
OUT = Path('data/mcf_features.pkl')
OUT.parent.mkdir(exist_ok=True)
df.to_pickle(OUT)
print(f'Wrote {OUT} ({OUT.stat().st_size / 1e6:.1f} MB)')

## Next: launch the dashboard

```bash
streamlit run app/app.py
```

Page 1 should orient you in five seconds; Page 2 surfaces the
non-obvious signals; Page 3 is the operational summary. Every chart
has an `ℹ️ How to read this chart` expander.